# CIFAR-10 Ensemble with 5 CNNs

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tensorchiefs/dl_course_2025/blob/main/notebooks/11_cifar10_ensemble_keras3_torch.ipynb)

In this notebook you will download the cifar10 dataset which contains quite small images (32x32x3) of 10 classes. The data is from the Canadian Institute For Advanced Research. After loading the dataset you will train 5 Models and use them as an ensemble. You will see that the ensemble model is always better or equal to the average of the indivual models

**Dataset:**  You work with the Cifar10 dataset. You have 60'000 32x32 pixel color images of 10 classes ("airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck")

**Content:**
* train 5 CNNS on Cifar10
* Use ensemble of models to predict on the testset

In [2]:
import keras
keras.config.set_backend("torch")  # Ensure we're using the torch backend

import numpy as np
import matplotlib.pyplot as plt
from keras import layers, ops, Model
from keras.datasets import cifar10
from keras.utils import to_categorical
from sklearn.metrics import log_loss, accuracy_score
from tqdm import tqdm


In [3]:
# Load and preprocess CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step


In [4]:
# CNN model definition
def build_cnn():
    inputs = layers.Input(shape=(32, 32, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    model = Model(inputs, outputs)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


In [5]:
# Train 5 models
models = []
histories = []
for i in range(5):
    print(f"Training model {i+1}/5")
    model = build_cnn()
    history = model.fit(x_train, y_train_cat, epochs=10, batch_size=64,
                        validation_split=0.1, verbose=2)
    models.append(model)
    histories.append(history)


Training model 1/5
Epoch 1/10
704/704 - 9s - 13ms/step - accuracy: 0.4750 - loss: 1.4758 - val_accuracy: 0.5672 - val_loss: 1.2245
Epoch 2/10
704/704 - 6s - 9ms/step - accuracy: 0.6200 - loss: 1.0904 - val_accuracy: 0.6510 - val_loss: 1.0210
Epoch 3/10
704/704 - 7s - 10ms/step - accuracy: 0.6664 - loss: 0.9614 - val_accuracy: 0.6668 - val_loss: 0.9723
Epoch 4/10
704/704 - 6s - 9ms/step - accuracy: 0.7007 - loss: 0.8640 - val_accuracy: 0.7028 - val_loss: 0.8915
Epoch 5/10
704/704 - 7s - 11ms/step - accuracy: 0.7213 - loss: 0.7994 - val_accuracy: 0.6832 - val_loss: 0.9392
Epoch 6/10
704/704 - 6s - 9ms/step - accuracy: 0.7420 - loss: 0.7417 - val_accuracy: 0.7032 - val_loss: 0.8817
Epoch 7/10
704/704 - 7s - 10ms/step - accuracy: 0.7629 - loss: 0.6825 - val_accuracy: 0.7094 - val_loss: 0.8632
Epoch 8/10
704/704 - 6s - 9ms/step - accuracy: 0.7789 - loss: 0.6358 - val_accuracy: 0.7194 - val_loss: 0.8509
Epoch 9/10
704/704 - 7s - 9ms/step - accuracy: 0.7947 - loss: 0.5870 - val_accuracy: 0.71

In [6]:
# Get individual model predictions
probs_list = []
for model in models:
    probs = model.predict(x_test, verbose=0)
    probs_list.append(probs)

# Average predictions
probs_ensemble = np.mean(probs_list, axis=0)

# Ensemble evaluation
y_pred_ensemble = np.argmax(probs_ensemble, axis=1)
nll_ensemble = log_loss(y_test, probs_ensemble)
acc_ensemble = accuracy_score(y_test, y_pred_ensemble)
print(f"Ensemble NLL: {nll_ensemble:.4f}")
print(f"Ensemble Accuracy: {acc_ensemble:.4f}")


Ensemble NLL: 0.7542
Ensemble Accuracy: 0.7444


In [7]:
# Evaluate individual models
nlls = []
accs = []
for probs in probs_list:
    y_pred = np.argmax(probs, axis=1)
    nll = log_loss(y_test, probs)
    acc = accuracy_score(y_test, y_pred)
    nlls.append(nll)
    accs.append(acc)

print(f"Average Individual NLL: {np.mean(nlls):.4f}")
print(f"Average Individual Accuracy: {np.mean(accs):.4f}")


Average Individual NLL: 0.8964
Average Individual Accuracy: 0.7032


### Summary:
- This notebook trains 5 CNNs on CIFAR-10 using Keras 3 with the PyTorch backend.
- The ensemble model averages the predicted probabilities and achieves performance at least as good as the average of the individual models.
